# Plato Input Catalogue (PIC) 2.0.0.1

This notebook explores the PIC 2.0.0.1 catalogue and its sub-PICs. Ultimately the goal of this work is to produce feather-format files that can used by `picsim` to generate on-demand stellar catalogues to be used by `platonium`.

**Prerequsites:** 

To run this notebook you need to download the PIC 2.0.0.1 and  place following files alonside this notebook:

- `pLOPS2PIC2.0.0.1.csv`
- `pLOPN1PIC2.0.0.1.csv`

**Documentation:** 

The following technical notes helps you understand the PIC:

- `PLATO-DLR-PL-LI-0015_i4.4`
- `PLATO-DLR-PL-RP-0001_i4.3`
- `PLATO_SCI_UPD_TN_0019`
- `PLATO_SCI_UPD_TN_0020`
- `PLATO-SSDC-PDC-TN-0006`

**Last checked:** 

This notebook was last functional with the PlatoSim version:

`PlatoSim 3.7.0-191-g42fc64d4`

In [2]:
%load_ext autoreload
%autoreload 2
%matplotlib notebook

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
# Built-in
import os
import sys
import glob

# PlatoSim default
import scipy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# PLATOnium default
from astropy import units as u
from astropy.coordinates import SkyCoord
from pathlib import Path
from tqdm import tqdm 

# PlatoSim functions
import platosim.plot      as pt
import platosim.starquery as sq
import platosim.utilities as ut
from platosim.simfile      import SimFile
from platosim.simulation   import Simulation
from platosim.lightcurve   import LightCurve
from platosim.matplotlibrc import setup_paper
setup_paper(warning=False)

from IPython.display import display, HTML
display(HTML("<style>.container {width:90% !important; }</style>"))

## Functions

The following function is used to load-in the PIC ascii catalogue and generate a new catalogue in a binary format called feather.

In [ ]:
def loadNumpyPIC110(inputFileTar, inputFileCon):
    """Function to load PIC110 numpy binary catalogue. 

    This is a debrecated function used prior to PlatoSim 3.6.0.
    The columns loaded below are the following:

    PICidDR1   : PIC-ID-DR1 from Gaia DR2
    ra         : ICRS RA
    decl       : ICRS Dec
    gaiaV      : De-reddened V mag from Gaia colour photometry
    sampleFlag : Bitmaskdefining PIC samples
    teff       : Stellar effective temperature [K]
    radius     : Stellar radius [R_sun]
    mass       : Stellar mass [M_sun]
    nCameraObs : EOL number of cameras seeing the star
    """

    # TARGETS
    
    df = pd.DataFrame()
    pic_tar = np.load(inputFileTar)
    df['PIC']    = pic_tar[:,0].astype(float).astype(int)
    df['ra']     = pic_tar[:,1].astype(np.float64)
    df['dec']    = pic_tar[:,2].astype(np.float64)
    df['mag']    = pic_tar[:,3].astype(np.float64)
    df['sample'] = pic_tar[:,4].astype(float).astype(int)
    df['Teff']   = pic_tar[:,5].astype(np.float64)
    df['R']      = pic_tar[:,6].astype(np.float64)
    df['M']      = pic_tar[:,7].astype(np.float64)
    df['ncams']  = pic_tar[:,8].astype(float).astype(int)
    df['field']  = pic_tar[:,9].astype(str)
    df = df.iloc[1:]
    df = df.reset_index(drop=True)

    # CONTAMINANTS
    
    dc = pd.DataFrame()
    pic_con = np.load(inputFileCon)
    dc['PIC'] = pic_con[:,0].astype(float).astype(int)
    dc['ra']  = pic_con[:,1].astype(float)
    dc['dec'] = pic_con[:,2].astype(float)
    dc['mag'] = pic_con[:,4].astype(float)
    dc['dis'] = pic_con[:,3].astype(float)
    
    return df, dc

In [ ]:
def createPIC200(path):
    """Create the PIC200 feather files used by 'picsim'.

    This function loads the original PIC 2.0.0 catalogue that contains
    the PIC targets and stellar contaminant. It then selects the 
    right columns and saves them to a binary feather file. Note that
    the original input folders needs to be parsed.

    Execution
    ---------
    >> createPIC110(<path/to/pLOPS2PIC2.0.0.1-t>)
    >> createPIC110(<path/to/pLOPN1PIC2.0.0.1-t>)
    """

    # PIC TARGETS

    # Load ascii catalogue
    data = np.genfromtxt(f'{path}/filename.csv', delimiter=',',
                        usecols=[0, 3, 5, 53, 55, 56, 58, 60, 71])
    df = pd.DataFrame()
    df['PIC']    = data[:,0].astype(float).astype(int)
    df['ra']     = data[:,1].astype(np.float64)
    df['dec']    = data[:,2].astype(np.float64)
    df['mag']    = data[:,3].astype(np.float64)
    df['Teff']   = data[:,5].astype(np.float64)
    df['R']      = data[:,6].astype(np.float64)
    df['M']      = data[:,7].astype(np.float64)
    df['ncams']  = data[:,8].astype(float).astype(int)
    df['sample'] = data[:,4].astype(float).astype(int)

    # String field needs to be loaded seperately: PLATO field: N=North, S=South
    df['field'] = np.loadtxt(inputFileTar.with_suffix('.csv'), delimiter=',',
                             usecols=[68], dtype=str)

    # Drop nan rows
    df = df.dropna()

    # Store columns
    col_sample = df['sample']
    df = df.drop(columns=['sample'])
    df['sample'] = col_sample

    # Select PIC sample
    # NOTE 2 is stated in documentation but 3 is correct..
    df['sample'] = df['sample'].replace([1, 3, 4, 8], ['P1', 'P2', 'P4', 'P5'])

    # Change camera numbers
    df['ncams'] = df['ncams'].replace([5, 11, 16, 17, 22], [6, 12, 18, 18, 24])

    # Convert V Jonhson-Cousin to P passband
    df['mag'] = ut.passbandConversionV2P(df.mag, df.Teff)

    # Select catalogues
    ds = df[df.field == 'S']
    dn = df[df.field == 'N']

    # Drop field before saving
    ds = ds.drop(columns=['field'])
    dn = dn.drop(columns=['field'])

    # Reset indices
    ds = ds.reset_index(drop=True)
    dn = ds.reset_index(drop=True)

    # Save to feather files
    ds.to_feather('PIC200_LOPS2_targets.ftr')
    dn.to_feather('PIC200_LOPN1_targets.ftr')

    # PIC CONTAMINATS

    # Load data
    # NOTE The PIC is the target to which the contaminants refers to
    data = np.loadtxt(inputFileCon.with_suffix('.csv'), delimiter=',',
                      skiprows=1, usecols=[2, 6, 8, 5, 19])
    dc =pd.DataFrame()
    dc['PIC'] = data[:,0].astype(float).astype(int)
    dc['ra']  = data[:,1].astype(np.float64)
    dc['dec'] = data[:,2].astype(np.float64)
    dc['mag'] = data[:,4].astype(np.float64)
    dc['dis'] = data[:,3].astype(float).astype(int)

    # Convert Vmag to Pmag using host star Teff
    # NOTE assumption needed for PlatoSim!
    for i in tqdm(range(len(dc)), bar_format=ut.tqdmBar()):
        df_i = df[df.PIC == dc.PIC.iloc[i]]
        dc.mag.iloc[i] = ut.passbandConversionV2P(dc.mag.iloc[0], df_i.Teff)

    # Sort after pointing
    ds = df[(ds.dec < 0)]
    dn = df[(df.dec > 0)]

    # Save to feather files
    ds.to_feather('PIC200_LOPS2_contaminants.ftr')
    dn.to_feather('PIC200_LOPN1_contaminants.ftr')

## Create target and contaminant feather catalogues

In [ ]:
createPIC200('pLOPS2PIC2.0.0.1-t')

In [ ]:
createPIC200('pLOPS2PIC2.0.0.1-t')

That's it! The output feather-format catalogues are placed on the KU Leuven FTP located at `/STER/platoman/ftp`. Located there `picsim` automatically downloads these files upon first execution and you can enjoy the user-friendly interface of `picsim` to generate your sub-PIC catalogues used by `platonium`.